In [5]:
#批量读书
import pandas as pd
import json
import chardet
from time import sleep
import time
import requests
import pandas as pd
import random
import pymysql
import json
import sys
import sqlalchemy
from datetime import datetime,date, timedelta
import os
import openai
from pathlib import Path
from sqlalchemy import text
from datetime import datetime
import shutil
import pandas as pd
import re
import time

# 阿里云 Qwen-Long API 配置
#  api_key="sk-f4f4de7b9a3f4dbcbab21012757d4fca"
api_key ="sk-3236ebaa10bc477987e3264e8a8b7d0f"

base_url = 'https://dashscope.aliyuncs.com/compatible-mode/v1'
client = openai.OpenAI(api_key=api_key, base_url=base_url)


def upload_file_to_qwen(file_path):
    file_object = client.files.create(file=Path(file_path), purpose="file-extract")
    return file_object.id

def summarize_content(file_id):
    list=client.files.list()
    list1=list.data
    status=0
    while status==0:
        for item in list1:
            if item.id == file_id:
                status = item.status
        if status =='processed':
            status=1
        time.sleep(10)
    try:
        completion = client.chat.completions.create(
            model="qwen-long",
            messages=[
                {'role': 'system', 'content': 'You are a helpful assistant.'},
                {'role': 'system', 'content': f'fileid://{file_id}'},
                {'role': 'user', 'content': '''
                作为高级财务信息筛选师，您的任务是依据严格的财务准则，精准提取并总结财务报表摘要中不同年份的两个核心财务板块信息：股东分红与资本公积增加，务必注重以下细化准则：
                ###股东分红:
                * “项目”列统一标记为“股东分红”，按年份求和汇总。
                * 纳入统计：分配、应付、提取、派发的股利、分配红利、分配现金股利、分配普通股股利、分配优先股股利、上缴收入、上缴利润、上缴股利、其他权益工具股利等类似的现金股息发放表述对应金额，或者类似分配利润、持有者分配、永续债付息、永续债利息、可续期公司债利息这类表述也算。
                * 不考虑“股票股利”。
                * 财报可能不将其他权益工具股利、永续债付息、、永续债利息、可续期公司债利息算作股东分红，但我们也需要单独统计这类分红。
                * “增加原因”列记录分红的具体形式。
                ###资本公积增加:
                * 范围：仅考虑资本公积增加、增长，不考虑资本公积减少的类目；“减少”都不纳入统计和考虑。
                * “项目”列统一为：资本公积增加；
                * “增加原因”列记录增加的主要原因，如"政府现金拨款"、"股权无偿划转"、"资产评估增值"、"项目移交"等；对于没有明确记录增加原因的，标记为"其他"；“增加原因”列不能为"资本公积增加"。
                * 同一年份可以有不同的 “增加原因”，但同一年份的同类 “增加原因”需要求和汇总，例如同一年份的 “增加原因”：其他只能有一条。
                * 该部分不允许使用"股东分红",避免产生混淆。 
                * 盈余公积、股本、注册资本、其他综合收益、其他收益、股权认缴、增资扩股、递延收益、其他权益工具，这些大类之下的内容都不算资本公积，资本公积必须严格包括“资本公积”四个字，需要将提取信息范围严格限制在“资本公积”大类之下。
                * 没有增加即0，即没有资本公积增加，此时就不用考虑其他资本公积信息了。
                * 财报表格中只有明确列名为“增加”、“减少”才值变化值，其他默认都为余额概念。不能把余额当做变化。
                ###时间标识与解析准则：
                *  “2023年”相关表述（如当期/当年/本年/今年）均指2023年度数据。
                *  “2022年”相关（上一期/上年/去年/上期）均指2022年度数据。 
                * 具体年份直接参照其字面意义，如“2020年”。
                * 注意区分文字表达与实际财务年度的一致性。
                ###金额处理原则：
                * 单位转换：“原始金额”列保留数字，例如：12345；“原始金额单位”列展示其原始单位，例如：元；金额列仅保留数值，所有数值都转换为“亿”单位（不展示单位），确保金额单位从“元”转换到“亿”时，准确地执行每一位数字的转换；“元”转“亿”除以100000000，“万”与“百万”分别除以10000和100转换为“亿”；“千元”除以100000转换为“亿”；已为“亿”的数值保持不变。 - **示例**：确保78,387,466.98元准确转为0.783875亿，保证转换精度无误。
                ###数据匹配准确性。
                * 确保每项财务事件与相应年份精确匹配，例如，2020年IPO筹集资金归档于2020年记录中。
                ###输出格式：
                ``` 年份|项目|增加原因|金额|原始金额|原始金额单位  …|…|…|…|…|…``` - 精准遵循该表格格式，需要带表头，不得改动表头的任何字符（年份|项目|增加原因|金额|原始金额|原始金额单位），不得遗漏符合特征的内容，确保年份至少包括2023和2022，以及年份对应的股东分红，无匹配数据时以"0"表示。

                请依据上述极其具体和明确的指导，审视财务报告摘录，实现股东分红与现金注资数据的高度精确抽取作业，重点关注现金注资识别的精确性，并确保在最终单位转换上的无误呈现。'''}
            ],
            stream=False
        )
        return completion.choices[0].message.dict()['content'].strip()
    except Exception as e:
        print(f"Error processing file {file_id}: {e}")
        return ''

def upload_metadata_to_mysql(file_id, filename, summary):
    sql_engine = sqlalchemy.create_engine(
    'mysql+pymysql://%s:%s@%s:%s/%s' % (
        'hz_work',
        'Hzinsights2015',
        'rm-uf6c8yi6zdq6ea7p1qo.mysql.rds.aliyuncs.com',
        '3306',
        'yq',
    ), poolclass=sqlalchemy.pool.NullPool
    )
    _cursor = sql_engine.connect()
    sql_engine1 = sqlalchemy.create_engine(
    'mysql+pymysql://%s:%s@%s:%s/%s' % (
        'root',
        's5bx55dh',
        'bja.sealos.run',
        '44525',
        'timinfo',
    ), poolclass=sqlalchemy.pool.NullPool
    )
    _cursor1 = sql_engine1.connect()

    date = datetime.now()
    # _cursor.execute(
    #     "INSERT INTO documents (dt, fileid, summary) VALUES (%s, %s, %s)",
    #     (date,file_id, summary)
    # )

    sql = """
    INSERT ignore INTO `6` (dt, fileid, filename,summary)
    VALUES (:dt, :fileid, :filename, :summary)
    """
    # 构建参数字典
    params = {
        "dt": date,
        "fileid": file_id,
        "filename": filename,
        "summary":summary
    }
    _cursor1.execute(text(sql),params)
    _cursor1.commit()
    _cursor1.close()
    print(params)

def insert_fi(df):
  with sql_engine.begin() as connection:
    df.to_sql('23年财报挖掘',connection,if_exists='append',index=False)

sql_engine = sqlalchemy.create_engine(
  'mysql+pymysql://%s:%s@%s:%s/%s' % (
    'hz_work',
    'Hzinsights2015',
    'rm-uf6c8yi6zdq6ea7p1qo.mysql.rds.aliyuncs.com',
    '3306',
    'yq',
  ), poolclass=sqlalchemy.pool.NullPool
  )


def pro_sum(summary):

    pattern = re.compile(r'```(.*)```', re.MULTILINE | re.DOTALL)
    match = pattern.search(summary)
    if match:
        summary = match.group(1)
    summary=summary.replace('-','').replace(' ','').replace('空值','')
    try:
        # 分割字符串为行
        lines = summary.split('\n')
        lines = [line for line in lines if line.strip()]
        # 移除标题和注释行
        header = lines[0].split('|')
        data_lines = [line.split('|') for line in lines[1:]]
        # 创建DataFrame
        df = pd.DataFrame(data_lines, columns=header)
        # 清理数据
        df = df.apply(lambda x: x.str.strip())  # 去除多余的空格
        df=df[df['年份']!='']
    except:
        df=pd.DataFrame()
    return summary,df

def process_files(folder_path,folder_path1,folder_path2,folder_path3):
    sql="""
    select distinct fileName
    from yq.`23年财报挖掘`
    where summary !=''
    """
    with sql_engine.begin() as connection:
        df_dbfile = pd.read_sql(sql, connection)
    list_dbfile=df_dbfile['fileName'].tolist()

    sql="""
    SELECT trade_code,fileName,
    公司名称 as company
    from yq.23年财报文件
    where fileName !=''
    """

    with sql_engine.begin() as connection:
        df_basicinfo = pd.read_sql(sql, connection)
    file_list=os.listdir(folder_path)
    total_files=len(file_list)
    processed_files=1
    for filename in file_list:
        print(f'---{processed_files}/{total_files}')
        processed_files+=1
        filename0=filename.replace('处理_','')
        file_path = os.path.join(folder_path, filename)
        if filename0 in list_dbfile:
            os.remove(file_path)
            continue
        if filename0 not in df_basicinfo['fileName'].tolist():
            shutil.copy(file_path, folder_path2)
            if os.path.exists(folder_path2):
                os.remove(file_path)
            else:
                print(r"{filename}数据库中不存在，且复制文件失败，未执行删除原文件操作。")
            continue
        company=df_basicinfo[df_basicinfo['fileName']==filename0]['company'].iloc[0]
        trade_code=df_basicinfo[df_basicinfo['fileName']==filename0]['trade_code'].iloc[0]
        
        if filename.endswith('.pdf'):
            # try:
                # 上传文件到阿里云
            list=client.files.list()
            list1=list.data
            file_exist=0
            for item in list1:
                if item.filename == filename:
                    file_id=item.id
                    file_exist = 1

            if file_exist==1:
                try:
                    summary = summarize_content(file_id)
                    # print(summary)
                except:
                    print('上传失败')
                    time.sleep(30)
                    shutil.copy(file_path, folder_path3)
                    if os.path.exists(folder_path3):
                        os.remove(file_path)
                    else:
                        print(r"{filename}上传失败，且复制文件失败，未执行删除原文件操作。")
                    continue
            else:
                try:
                    file_id = upload_file_to_qwen(file_path)
                    # 获取文件总结
                    summary = summarize_content(file_id)
                    # print(summary)
                except:
                    print('上传失败')
                    time.sleep(30)
                    shutil.copy(file_path, folder_path3)
                    if os.path.exists(folder_path3):
                        os.remove(file_path)
                    else:
                        print(r"{filename}上传失败，且复制文件失败，未执行删除原文件操作。")
                    continue
            summary0,df=pro_sum(summary)
            if df.empty:
                data = {
                    'trade_code': [trade_code],
                    'fileName': [filename0],
                    '公司名称': [company],
                    '年份':[2023],
                    '项目':['股东分红'],
                    '增加原因':['现金分红'],
                    '金额':[0],
                    '原始金额':[0],
                    '备注': ['提取失败'],
                    'summary':[summary]
                }
                df = pd.DataFrame(data)
            else:
                df['trade_code']=trade_code
                df['fileName']=filename0
                df['公司名称']=company
                df['备注']='提取成功'
                df['summary']=summary
            try:
                insert_fi(df)
                sleep(10)
                # 删除本地文件
                shutil.copy(file_path, folder_path1)
                if os.path.exists(folder_path1):
                    os.remove(file_path)
                else:
                    print("复制文件失败，未执行删除原文件操作。")
            except:
                shutil.copy(file_path, folder_path2)
                if os.path.exists(folder_path2):
                    os.remove(file_path)
                else:
                    print("操作失败失败，未执行删除原文件操作。")
            # except Exception as e:
            #     print(f"Error processing file {filename}: {e}")

def main():
    folder_path = r"D:\2024年\企业预警通\新建图片目录"
    folder_path1 = r"D:\2024年\已处理"
    folder_path2=r"D:\2024年\名称与数据库不匹配"
    folder_path3=r"D:\2024年\上传失败"
    os.makedirs(folder_path, exist_ok=True)
    os.makedirs(folder_path1, exist_ok=True)
    os.makedirs(folder_path2, exist_ok=True)
    os.makedirs(folder_path3, exist_ok=True)

    process_files(folder_path,folder_path1,folder_path2,folder_path3)
    # process_files(folder_path1)

if __name__ == '__main__':
    main()

# start_marker = 'fileid://'
# if inputs.startswith('fileid://'):
#     start_index = inputs.find(start_marker) + len(start_marker)
#     prompt_text1 = 'You are a helpful assistant.'
#     prompt_text2 = inputs[:39]




---1/121
---2/121
---3/121
---4/121
---5/121
---6/121
---7/121
---8/121
---9/121
---10/121
---11/121
---12/121
---13/121
---14/121
---15/121
---16/121
---17/121
---18/121
---19/121
---20/121
---21/121
---22/121
---23/121
---24/121
---25/121
---26/121
---27/121
---28/121
---29/121
---30/121
---31/121
---32/121
---33/121
---34/121
---35/121
---36/121
---37/121
---38/121
---39/121
---40/121
---41/121
---42/121
---43/121
---44/121
---45/121
---46/121
---47/121
---48/121
---49/121
---50/121
---51/121
---52/121
---53/121
---54/121
---55/121
---56/121
---57/121
---58/121
---59/121
---60/121
---61/121
---62/121
---63/121
---64/121
---65/121
---66/121
---67/121
---68/121
---69/121
---70/121
---71/121
---72/121
---73/121
---74/121
---75/121
---76/121
---77/121
---78/121
---79/121
---80/121
---81/121
---82/121
---83/121
---84/121
---85/121
---86/121
---87/121
---88/121
---89/121
---90/121
---91/121
---92/121
---93/121
---94/121
---95/121
---96/121
---97/121
---98/121
---99/121
---100/121
---101/1

In [10]:
df=pd.DataFrame()
# df['trade_code']='261136.SH'
# df['fileName']='海尔金融保理（重庆）有限公司2023年度审计报告.pdf'
# df['公司名称']='海尔金融保理(重庆)有限公司'
# df['备注']='提取失败'
# print(df)
if df.empty:
    data = {
        'trade_code': ['261136.SH'],
        'fileName': ['海尔金融保理（重庆）有限公司2023年度审计报告.pdf'],
        '公司名称': ['海尔金融保理(重庆)有限公司'],
        '年份':[2023],
        '项目':['股东分红'],
        '金额':[0],
        '备注': ['提取失败'],
        'summary':['1']
    }
    df = pd.DataFrame(data)
print(df)


  trade_code                      fileName            公司名称    年份    项目  金额  \
0  261136.SH  海尔金融保理（重庆）有限公司2023年度审计报告.pdf  海尔金融保理(重庆)有限公司  2023  股东分红   0   

     备注 summary  
0  提取失败       1  


In [5]:
a=73540000
b=98379000
a+b
1.71919

171919000

In [10]:
import pandas as pd
import re
summary='''```
年份|项目|金额
||
2023|股东分红|4.868576
2023|现金注资|2.360553
2022|股东分红|0.140192
2022|现金注资|空值
```

解释：
2023年股东分红为486,857,641.55元，转换为亿元是4.868576亿元；
2023年现金注资是根据资本公积增加项中无偿划转它山堰文化旅游发展有限公司部分股权的批复，增加资本公积236,055,343.21元，转换为亿元是2.360553亿元；
2022年股东分红为14,019,173.29元，转换为亿元是0.140192亿元；
2022年现金注资没有提及具体数值，因此使用空值表示。'''
pattern = re.compile(r'```(.*)```', re.MULTILINE | re.DOTALL)

match = pattern.search(summary)

if match:
    table_content = match.group(1)
    print(table_content)
summary=table_content.replace('-','').replace(' ','').replace('空值','')

# 分割字符串为行
lines = summary.split('\n')
# 使用列表推导式过滤空行
lines = [line for line in lines if line.strip()]
# 移除标题和注释行
header = lines[0].split('|')
data_lines = [line.split('|') for line in lines[1:]]
# 创建DataFrame
df = pd.DataFrame(data_lines, columns=header)
# 清理数据
df = df.apply(lambda x: x.str.strip())  # 去除多余的空格
df=df[df['年份']!='']
df


年份|项目|金额
||
2023|股东分红|4.868576
2023|现金注资|2.360553
2022|股东分红|0.140192
2022|现金注资|空值



,年份,项目,金额
1,2023,股东分红,4.868576
2,2023,现金注资,2.360553
3,2022,股东分红,0.140192
4,2022,现金注资,


In [9]:
lines[0].split('|')


['年份｜项目｜金额']

In [2]:
#特定书籍
import pandas as pd
import json
import chardet
from time import sleep
import time
import requests
import pandas as pd
import random
import pymysql
import json
import sys
import sqlalchemy
from datetime import datetime,date, timedelta
import os
import openai
from pathlib import Path
from sqlalchemy import text
from datetime import datetime
import shutil

# 阿里云 Qwen-Long API 配置
api_key = api_key="sk-f4f4de7b9a3f4dbcbab21012757d4fca"
base_url = 'https://dashscope.aliyuncs.com/compatible-mode/v1'
client = openai.OpenAI(api_key=api_key, base_url=base_url)


def upload_file_to_qwen(file_path):
    file_object = client.files.create(file=Path(file_path), purpose="file-extract")
    return file_object.id

def summarize_content(file_id):
    list=client.files.list()
    list1=list.data
    status=0
    while status==0:
        for item in list1:
            if item.id == file_id:
                status = item.status
        if status =='processed':
            status=1
        time.sleep(10)
    try:
        completion = client.chat.completions.create(
            model="qwen-long",
            messages=[
                {'role': 'system', 'content': 'You are a helpful assistant.'},
                {'role': 'system', 'content': f'fileid://{file_id}'},
                {'role': 'user', 'content': '严格遵照以下框架总结全书内容,至少需要包括每章的名称： 1. 章节介绍： - 阐明各章主题及其在整体结构中的位置与贡献。概括每个章节的核心结论，并阐述作者利用何种证据、案例研究或理论分析支撑这些结论，确保逻辑线索清晰连贯。 2. 内容全面性： - 确保概要覆盖文献的所有关键章节与小节，无重要信息遗漏，全面反映书籍的主体内容与细节。 3. 格式标准化： - 采用统一格式记录每章每节的概要，参考模板如下： ``` 第X章：[章节标题] - 主要结论：... - 分析路径：... ``` 4. 总体归纳： - 文档最后，汇总整本书的关键发现、主要贡献及作者的论述逻辑，突出其在相应学术领域或实践中的价值与影响力，强化概要的总结性与指导意义。。'}
            ],
            stream=False
        )
        return completion.choices[0].message.dict()['content'].strip()
    except Exception as e:
        print(f"Error processing file {file_id}: {e}")
        return ''

def upload_metadata_to_mysql(file_id, filename, summary):
    sql_engine = sqlalchemy.create_engine(
    'mysql+pymysql://%s:%s@%s:%s/%s' % (
        'hz_work',
        'Hzinsights2015',
        'rm-uf6c8yi6zdq6ea7p1qo.mysql.rds.aliyuncs.com',
        '3306',
        'yq',
    ), poolclass=sqlalchemy.pool.NullPool
    )
    _cursor = sql_engine.connect()
    sql_engine1 = sqlalchemy.create_engine(
    'mysql+pymysql://%s:%s@%s:%s/%s' % (
        'root',
        's5bx55dh',
        'bja.sealos.run',
        '44525',
        'timinfo',
    ), poolclass=sqlalchemy.pool.NullPool
    )
    _cursor1 = sql_engine1.connect()

    date = datetime.now()
    # _cursor.execute(
    #     "INSERT INTO documents (dt, fileid, summary) VALUES (%s, %s, %s)",
    #     (date,file_id, summary)
    # )

    sql = """
    INSERT ignore INTO `书库` (dt, fileid, filename,summary)
    VALUES (:dt, :fileid, :filename, :summary)
    """
    # 构建参数字典
    params = {
        "dt": date,
        "fileid": file_id,
        "filename": filename,
        "summary":summary
    }
    _cursor1.execute(text(sql),params)
    _cursor1.commit()
    _cursor1.close()
    print(params)


def process_files(folder_path,folder_path1):
    for filename in os.listdir(folder_path):
        file_path = os.path.join(folder_path, filename)
        if filename.endswith('.pdf'):
            # try:
                # 上传文件到阿里云
            list=client.files.list()
            list1=list.data
            file_exist=0
            for item in list1:
                if item.filename == filename:
                    file_exist = 1

            if file_exist==1:
                continue
            file_id = upload_file_to_qwen(file_path)
            print(f'{file_id}  {filename}')
            # 获取文件总结
            # summary = summarize_content(file_id)
            
            # 将文件ID和总结内容上传到MySQL
            # upload_metadata_to_mysql(file_id, filename,summary)
            # 删除本地文件
            shutil.copy(file_path, folder_path1)
            if os.path.exists(folder_path1):
                os.remove(file_path)
            else:
                print("复制文件失败，未执行删除原文件操作。")
            # except Exception as e:
            #     print(f"Error processing file {filename}: {e}")

def main():
    folder_path = r"C:\Users\Administrator\WPSDrive\389717562\WPS云盘\WPS\新建文件夹"
    folder_path1 = r"C:\Users\Administrator\WPSDrive\389717562\WPS云盘\WPS\lib\书籍"
    
    # process_files(folder_path,folder_path1)
    # process_files(folder_path1)

if __name__ == '__main__':
    main()

# start_marker = 'fileid://'
# if inputs.startswith('fileid://'):
#     start_index = inputs.find(start_marker) + len(start_marker)
#     prompt_text1 = 'You are a helpful assistant.'
#     prompt_text2 = inputs[:39]

In [3]:
#删除文件

from openai import OpenAI
from sqlalchemy import text
sql_engine = sqlalchemy.create_engine(
    'mysql+pymysql://%s:%s@%s:%s/%s' % (
        'hz_work',
        'Hzinsights2015',
        'rm-uf6c8yi6zdq6ea7p1qo.mysql.rds.aliyuncs.com',
        '3306',
        'yq',
    ), poolclass=sqlalchemy.pool.NullPool
    )
_cursor = sql_engine.connect()
sql_engine1 = sqlalchemy.create_engine(
    'mysql+pymysql://%s:%s@%s:%s/%s' % (
        'root',
        's5bx55dh',
        'bja.sealos.run',
        '44525',
        'timinfo',
    ), poolclass=sqlalchemy.pool.NullPool
    )
_cursor1 = sql_engine1.connect()
# sql="""SELECT t1.fileid
# FROM 信息库 t1
# JOIN (
#     SELECT filename
#     FROM 信息库
#     GROUP BY filename
#     HAVING COUNT(*) > 1
# ) duplicates ON t1.filename = duplicates.filename
# WHERE t1.fileid NOT IN (
#     SELECT MIN(tmp.fileid)
#     FROM 信息库 tmp
#     WHERE t1.filename = tmp.filename
#     GROUP BY tmp.filename
# )"""
# df=pd.read_sql(sql,_cursor1)
target_ids=['file-fe-0ZCtEYqRCIqsGV4rvutc0kVY']
# fileid='file-fe-0dFqE9WNplIZl2lhz5pgIVSg'
# client.files.delete(fileid)
for fileid in target_ids:
    client.files.delete(fileid)
    sql="""delete from 书库 where fileid='%s'"""%(fileid)
    _cursor1.execute(text(sql))
    _cursor1.commit()

NotFoundError: Error code: 404 - {'error': {'message': 'No such File object: file-fe-0ZCtEYqRCIqsGV4rvutc0kVY', 'type': 'invalid_request_error', 'param': 'id', 'code': None}}

In [10]:
import os

def collect_top_level_files_as_txt(folder_path):
    txt_files_list = []
    
    # 获取指定文件夹下的所有文件和子文件夹名称
    items = os.listdir(folder_path)
    
    # 遍历文件夹内的每个项
    for item in items:
        # 拼接完整路径
        full_item_path = os.path.join(folder_path, item)
        # 检查是否为文件（非目录）
        if os.path.isfile(full_item_path):
            # 构造新的文件名，替换或添加.txt后缀
            txt_file_path = os.path.splitext(full_item_path)[0] + ".txt"
            # 将构造后的.txt文件路径添加到列表中
            txt_files_list.append(txt_file_path)
    
    return txt_files_list

# 指定文件夹路径
folder_path = r'C:\Users\Administrator\WPSDrive\389717562\WPS云盘\WPS\quick1'
# 调用函数并打印结果
txt_files = collect_top_level_files_as_txt(folder_path)
list=client.files.list()
list1=list.data
for item in list1:
    if item.filename in txt_files:
        client.files.delete(item.fileid)

In [3]:
list=client.files.list()
list1=list.data
# status=0
# while status==0:
#     for item in list1:
#         if item['id'] == target_id:
#             status = item['status']
#     if status =='processed':
#         status=1
#     sys.wait(10)

# list=client.files.list()
# list1=list.data
# file_exitst=0
# for item in list1:
#     if item['filename'] == filename:
#         file_exitst = 1



        

In [9]:
list1[0].filename

'从10万份文档中更快、更准确地找到信息，还能理解语义！试试ElasticSearch+RAG070433_87u7bsor.txt'

In [5]:
#续问问题

file_id='file-fe-ANYSWI6wNhB2yFCR5t5bVCuX'
completion = client.chat.completions.create(
        model="qwen-long",
        messages=[
            {'role': 'system', 'content': f'You are a helpful assistant.'},
            {'role': 'system', 'content': f'fileid://{file_id}'},
            {'role': 'user', 'content': '''
            请编制一份详尽而精细的文献章节分析概要，着重于深度解析与结构完整性。概要需严格遵照以下框架制定： 1. **章节与小节细分**： - 对文献中的每一章节进行细致划分，阐明各章主题及其在整体结构中的位置与贡献。概括每个章节的核心结论，并阐述作者利用何种证据、案例研究或理论分析支撑这些结论，确保逻辑线索清晰连贯。 2. **内容全面性**： - 确保概要覆盖文献的所有关键章节与小节，无重要信息遗漏，全面反映书籍的主体内容与细节。 3. **格式标准化**： - 采用统一格式记录每章每节的概要，参考模板如下： ``` **第X章：[章节标题]** - **主要结论：**... - **分析路径：**... ``` 4. **总体归纳**： - 文档最后，汇总整本书的关键发现、主要贡献及作者的论述逻辑，突出其在相应学术领域或实践中的价值与影响力，强化概要的总结性与指导意义。 请依据上述准则，创造一份既深入又高度提炼的文献分析指南。'''}
        ],
        stream=False
    )
completion.choices[0].message.dict()['content'].strip()

'**第1章：导论**\n- **主要结论：**本章概述了研究的背景、文献综述、理论框架及研究方法，并介绍了全书的主要内容、篇章结构和创新亮点。作者强调了“一国两制”理论与实践研究的重要性和紧迫性，特别是在全球化和中国改革开放的大背景下。\n- **分析路径：**通过回顾改革开放40年的历程，分析了“一国两制”在中国特色社会主义理论体系中的位置，以及其在港澳地区的实施经验，为台湾问题提供参考。采用文献综述与理论分析相结合的方法，明确了研究的理论框架和实证路径。\n\n**第2章：“一国两制”在港澳地区的实践经验**\n- **主要结论：**此章详细探讨了“一国两制”在港澳地区的具体实施情况，包括初始环境、利益博弈、制度调适、经济政治环境、文化认同演变及未来发展方向。指出香港与澳门在实施中遇到的挑战与机遇各不相同，澳门相对香港对“一国两制”的认同度更高。\n- **分析路径：**通过案例分析，如CEPA协议、香港民主与法治建设等，结合实地调研和文献资料，分析了“一国两制”实践中的得与失，以及对台湾模式的启示。\n\n**第3章：台湾社会对“一国两制”模式的态度及其变化趋势**\n- **主要结论：**台湾民众对“一国两制”存在复杂态度，受政治、文化因素影响，存在“拒统”心理，但也有学者对“一国两制”持开放态度。台湾认同的多元性影响了对“一国两制”的接受度。\n- **分析路径：**采用民调数据、专家访谈和理论分析，探讨了台湾社会对“一国两制”的认知差异及其背后的原因，分析了国家认同的政治与文化属性。\n\n**第4章：美国对“一国两制”的认知偏差和消极评估**\n- **主要结论：**美国政府、学术界和媒体对“一国两制”普遍存在误解和负面评价，这源于认知偏差、意识形态差异和个人利益考量。\n- **分析路径：**结合作者在哈佛大学的访学经历，通过文献分析与学术访谈，揭示了美国各界对“一国两制”的认知现状及其成因。\n\n**第5章：“一国两制”台湾模式的理论和现实意义**\n- **主要结论：**本章从理论与实践角度探讨了“一国两制”台湾模式的可行性，强调了其理论建构的重要性，并分析了两岸关系中的推动与干扰因素，对和平统一持审慎乐观态度。\n- **分析路径：**通过比较研究、理论构建和对台湾社会的深入分析，提出了“一国两制”台湾模式的差异化设计，以及如何在制度差异和

In [7]:
#使用通义千问总结
import pandas as pd
import json
import chardet
from time import sleep
import time
import requests
import pandas as pd
import random
import pymysql
import json
import sys
import sqlalchemy
from datetime import datetime,date, timedelta
import os
import openai
from pathlib import Path
from sqlalchemy import text
from datetime import datetime
import smtplib
from email.mime.multipart import MIMEMultipart
from email.mime.multipart import MIMEMultipart
from email.mime.text import MIMEText
import shutil

# 邮件配置
SMTP_SERVER = 'smtp.qq.com'
SMTP_PORT = 465
EMAIL_USER = '917952467@qq.com'
EMAIL_PASSWORD = 'ineaylzljuqdbgai'
RECIPIENT_EMAIL = 'tim.ye@foxmail.com'

def send_email(subject, html_content):
    msg = MIMEMultipart('alternative')
    msg['Subject'] = subject
    msg['From'] = EMAIL_USER
    msg['To'] = RECIPIENT_EMAIL
    
    part = MIMEText(html_content, 'html')
    msg.attach(part)
    
    with smtplib.SMTP_SSL(SMTP_SERVER, SMTP_PORT) as server:
        server.connect(SMTP_SERVER, SMTP_PORT)
        server.login(EMAIL_USER, EMAIL_PASSWORD)
        server.sendmail(EMAIL_USER, RECIPIENT_EMAIL, msg.as_string())

# 阿里云 Qwen-Long API 配置
api_key = api_key="sk-f4f4de7b9a3f4dbcbab21012757d4fca"
base_url = 'https://dashscope.aliyuncs.com/compatible-mode/v1'
client = openai.OpenAI(api_key=api_key, base_url=base_url)


def upload_file_to_qwen(file_path):
    file_object = client.files.create(file=Path(file_path), purpose="file-extract")
    return file_object.id

def summarize_content(file_id):
    completion = client.chat.completions.create(
        model="qwen-long",
        messages=[
            {'role': 'system', 'content': 'You are a helpful assistant.'},
            {'role': 'system', 'content': f'fileid://{file_id}'},
            {'role': 'user', 'content': '请你帮我总结一下这篇文章的主要内容，请保证结构清晰、内容准确、逻辑通顺，并且确保不能遗漏重要内容，用中文回答'}
        ],
        stream=False
    )
    content = completion.choices[0].message.dict()['content']
    return content
def upload_metadata_to_mysql(file_id, summary,filename):
    sql_engine = sqlalchemy.create_engine(
    'mysql+pymysql://%s:%s@%s:%s/%s' % (
        'hz_work',
        'Hzinsights2015',
        'rm-uf6c8yi6zdq6ea7p1qo.mysql.rds.aliyuncs.com',
        '3306',
        'yq',
    ), poolclass=sqlalchemy.pool.NullPool
    )
    _cursor = sql_engine.connect()
    date = datetime.now()
    # _cursor.execute(
    #     "INSERT INTO documents (dt, fileid, summary) VALUES (%s, %s, %s)",
    #     (date,file_id, summary)
    # )

    sql = """
    INSERT ignore INTO 信息库 (dt, fileid, summary,filename)
    VALUES (:dt, :fileid, :summary,:filename)
    """
    # 构建参数字典
    params = {
        "dt": date,
        "fileid": file_id,
        "summary":summary,
        "filename":filename
    }
    _cursor.execute(text(sql),params)
    _cursor.commit()
    _cursor.close()
    filename=filename+'文件编号：'+file_id
    send_email(filename,summary)

def process_files(folder_path,folder_path2):
    for filename in os.listdir(folder_path):
        file_path = os.path.join(folder_path, filename)
        if filename.endswith('.pdf') or filename.endswith('.docx'):
            try:
                # 上传文件到阿里云
                file_id = upload_file_to_qwen(file_path)
                # 获取文件总结
                summary = summarize_content(file_id)
                
                # 将文件ID和总结内容上传到MySQL
                upload_metadata_to_mysql(file_id, summary,filename)
                
                # 删除本地文件
                shutil.move(file_path, folder_path2)
            except Exception as e:
                print(f"Error processing file {filename}: {e}")

def main():
    folder_path = r"C:\Users\Administrator\iCloudDrive\quickinfo"
    folder_path1 = r"C:\Users\Administrator\WPSDrive\389717562\WPS云盘\WPS\quick"
    folder_path2 = r"C:\Users\Administrator\WPSDrive\389717562\WPS云盘\WPS\lib"

    
    # process_files(folder_path,folder_path2)
    # process_files(folder_path1,folder_path2)

if __name__ == '__main__':
    main()


In [1]:
from tika import parser

# 解析PDF文件
parsed = parser.from_file(r"C:\Users\Administrator\WPSDrive\389717562\WPS云盘\WPS\lib\DM研究_起底存储半导体（一）：美光科技的白与黑.docx")

# 提取文本
text = parsed['content']

# 打印文本内容
print(text)


2024-06-27 20:59:32,807 [MainThread  ] [WARNI]  Failed to see startup log message; retrying...

































DMI研究|起底存储半导体 (一):美光科技的白与黑





【美元债主体系列报告】






对本文内容有交流合作等需求
请电话联系+852-3990-1786
或扫描下方二维码联系DMI 客服




 (
700.00
600
 
00
微
 
处
 
理
 
器
 
14%
集成
 
82
电路
%
500.00
传感器
 
4%
400.00
300.00
200.00
00.00
光电子
 
8%
模拟电路
 
15%
分
6%
逻提中
31%
0.00
■分立器件■光电子■传感器■模拟电路■微处理器■存储■逻
辑电路
·分业器件·光电子·传感器·模拟电路 微处理器·存储·逻辑电路
)


前述

全球半导体产业从上世纪四五十年代在美国起源后开始蓬勃发展，经历了划时代意义的技术革命、腥风血 雨的资本斗争，从美国到日本，从日本到韩国、中国台湾，再到中国大陆的三次产业区域转移后，当前行业格局 寡头特征明显，也由原先风投气息浓重的高速发展模式逐步转变成周期性明显的成熟行业，这给半导体债券市 场的后续发展可能性带来了必要条件。结合当下半导体债券市场来看，其整体规模由原先2008年的约28.62亿 美元增长到当前的4224.9亿美元，新发节奏于21年达到峰值，其中美元债占比约89.82%。分国家或地区来看， 美国企业在发债端占据全球主导地位，与其在全球市场供给端的地位相一致，历年比重维持在近七成以上。聚焦 美元债则情况更为突出，现存的367笔美元债中，美、中、日、韩及中国台湾合计占比近98.94%。而从二级市
场表现来看，选取的多支代表美元债(非可转债)最差收益率基本在6.5%附近，最新价格区间在【90,100】,
近一年价格走势在23年年初阶段有一定的下探情况，不同于其股价波动，整体表现较为稳定。

	
120000
	图1全球半导体债券分布及美元代表券价格走势
110

	 (
00
)
 (
80000
)
	 (
105
)
 (
100
)

	
 (
60000
)
 (
40000
)
	



	
	85

	20000
	

	
	80E


	 (
20082009201020112012201320142015201620172018201920202021
20222023


In [1]:
!pip install pyahocorasick

  Obtaining dependency information for pyahocorasick from https://files.pythonhosted.org/packages/36/76/d83c60ec7a202cbfeffaa9649d0fee6ddcb974622e411b86211ff3572549/pyahocorasick-2.1.0-cp311-cp311-win_amd64.whl.metadata


In [12]:
import tempfile
from pathlib import Path
import os

filename='DM研究_起底存储半导体（一）：美光科技的白与黑.docx'
# 使用os.path.splitext分离文件名和扩展名
base_name, extension = os.path.splitext(filename)
temp_file_path = tempfile.mkstemp(suffix=".txt", prefix=base_name + "_")[1]

with open(temp_file_path, "w+", encoding='utf-8') as temp_file:
    temp_file.write(text)


# 确保使用正确的purpose和根据实际情况调整
file_object = client.files.create(
    file=Path(temp_file_path),  # 使用Path对象指向临时文件
    purpose="file-extract",  # 根据您的需求调整目的
)

# 打印文件对象的ID或其他信息，根据需要
print(f"Uploaded file ID: {file_object.id}")
file_id=file_object.id
completion = client.chat.completions.create(
        model="qwen-long",
        messages=[
            {'role': 'system', 'content': 'You are a helpful assistant.'},
            {'role': 'system', 'content': f'fileid://{file_id}'},
            {'role': 'user', 'content': '请你帮我总结一下这篇文章的主要内容，请保证结构清晰、内容准确、逻辑通顺，并且确保不能遗漏重要内容，用中文回答'}
        ],
        stream=False
    )
content = completion.choices[0].message.dict()['content']
print(content)





Uploaded file ID: file-fe-7ghRD2V5HsNTZdaOK2Q3BThC
文章《DMI研究|起底存储半导体 (一):美光科技的白与黑》主要探讨了全球半导体产业的发展历程、当前格局、存储半导体领域的特点，以及美光科技公司在存储芯片行业的地位、财务表现和面临的挑战。以下是文章内容的结构化总结：

### 全球半导体产业概况
- **起源与发展**：半导体产业自20世纪四五十年代在美国兴起，经历多次技术革新和资本竞争，产业重心历经美国、日本、韩国、中国台湾至中国大陆的转移。
- **寡头垄断格局**：当前行业呈现明显的寡头特征，由初创时期的高风险投资模式转变为周期性强的成熟行业。
- **债券市场**：半导体债券市场规模大幅增长，从2008年的约28.62亿美元增长至4224.9亿美元，美元债占比约89.82%，美国企业为主导。

### 半导体产品分类与市场
- **集成电路核心地位**：集成电路占半导体产品销售额的80%以上，其中逻辑电路和存储IC（DRAM和NAND Flash）比重最高。
- **市场趋势**：存储板块份额虽有所下降，但仍为集成电路第二大细分市场，2023年预计存储芯片销售同比下降近35.24%。

### 存储芯片介绍
- **重要性**：在5G、云计算、AI等技术推动下，对存储芯片的需求剧增，数据量级提升至EB、ZB级。
- **类型**：主要分为RAM（如DRAM）和ROM（如NAND Flash），DRAM用于即时数据处理，NAND Flash则适合大容量、持久存储。

### 美光科技分析
- **市场地位**：美光在汽车存储市场接近50%份额，具有显著优势。
- **财务表现**：受供需关系影响，毛利率和营业利润率出现回调，FY23毛利率为-10.8%，营业利润率为-36.7%。
- **中国业务影响**：因中美科技限制，美光中国市场收入占比由2018年的58%降至2023年的14%，并面临网络安全审查导致的销售受限。
- **流动性与偿债能力**：公司现金资产比稳定在15%以上，现金比率超过1.5，优于同行，Net Debt/EBITDA表现良好。
- **未来发展**：行业龙头之一，拥有科研实力和战略背景，计划推出新产品抢占高端市场，有望受益于行业上行周期。

### 投资风险提示
- **声明**：报告

In [13]:
print(text)
































DMI研究|起底存储半导体 (一):美光科技的白与黑





【美元债主体系列报告】






对本文内容有交流合作等需求
请电话联系+852-3990-1786
或扫描下方二维码联系DMI 客服




 (
700.00
600
 
00
微
 
处
 
理
 
器
 
14%
集成
 
82
电路
%
500.00
传感器
 
4%
400.00
300.00
200.00
00.00
光电子
 
8%
模拟电路
 
15%
分
6%
逻提中
31%
0.00
■分立器件■光电子■传感器■模拟电路■微处理器■存储■逻
辑电路
·分业器件·光电子·传感器·模拟电路 微处理器·存储·逻辑电路
)


前述

全球半导体产业从上世纪四五十年代在美国起源后开始蓬勃发展，经历了划时代意义的技术革命、腥风血 雨的资本斗争，从美国到日本，从日本到韩国、中国台湾，再到中国大陆的三次产业区域转移后，当前行业格局 寡头特征明显，也由原先风投气息浓重的高速发展模式逐步转变成周期性明显的成熟行业，这给半导体债券市 场的后续发展可能性带来了必要条件。结合当下半导体债券市场来看，其整体规模由原先2008年的约28.62亿 美元增长到当前的4224.9亿美元，新发节奏于21年达到峰值，其中美元债占比约89.82%。分国家或地区来看， 美国企业在发债端占据全球主导地位，与其在全球市场供给端的地位相一致，历年比重维持在近七成以上。聚焦 美元债则情况更为突出，现存的367笔美元债中，美、中、日、韩及中国台湾合计占比近98.94%。而从二级市
场表现来看，选取的多支代表美元债(非可转债)最差收益率基本在6.5%附近，最新价格区间在【90,100】,
近一年价格走势在23年年初阶段有一定的下探情况，不同于其股价波动，整体表现较为稳定。

	
120000
	图1全球半导体债券分布及美元代表券价格走势
110

	 (
00
)
 (
80000
)
	 (
105
)
 (
100
)

	
 (
60000
)
 (
40000
)
	



	
	85

	20000
	

	
	80E


	 (
20082009201020112012201320142015201620172018201920202021
20222023
